# darija-bench — produire le négatif adversarial

Ce carnet fabrique le corpus qui manque au dépôt : de l'**arabe standard
écrit par un LLM sur des sujets du quotidien**.

## Pourquoi ce corpus

Le classifieur de `darija-lab` distingue très bien le tunisien du marocain
et de l'algérien. Il trébuche sur un registre précis : la **fusha
conversationnelle d'assistant** — 8,7 % de ces textes passent son seuil,
contre 0,3 % pour la prose encyclopédique de Wikipédia.

Ni Wikipédia ni les corpus maghrébins ne représentent ce registre. Il faut
donc le fabriquer.

## Pourquoi Colab

Il ne s'agit **pas** d'entraîner quoi que ce soit — le classifieur
s'entraîne en 46 secondes sur un portable. Il s'agit de faire **écrire** un
LLM, et les paliers gratuits d'API plafonnent à ~20 requêtes par jour.
Un GPU gratuit fait tourner un modèle ouvert localement : aucun quota.

## L'étiquette est propre

On demande explicitement au modèle de répondre **en fusha**. L'étiquette
vient donc de la consigne, jamais d'un score — sélectionner sur le score
ferait réapprendre au futur classifieur la frontière de l'actuel.

**Durée : 30 à 60 minutes.** Objectif : ~1 800 blocs de 60 mots.

## 1. Vérifier le GPU

Menu **Exécution → Modifier le type d'exécution → T4 GPU**, puis lancez la
cellule. Si elle affiche `AUCUN GPU`, le reste sera très lent.

In [ ]:
import torch
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
    print('memoire :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'Go')
else:
    print('AUCUN GPU — Execution > Modifier le type d execution > T4 GPU')

## 2. Installer

Deux minutes. Les avertissements de version sont normaux.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes
print('installe')

## 3. Charger le modèle

`Qwen2.5-7B-Instruct` : libre d'accès, sans autorisation à demander, et
correct en arabe. Quantifié en 4 bits pour tenir sur un T4.

Le chargement prend 3 à 5 minutes la première fois.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODELE = 'Qwen/Qwen2.5-7B-Instruct'

quant = BitsAndBytesConfig(load_in_4bit=True,
                           bnb_4bit_compute_dtype=torch.float16,
                           bnb_4bit_quant_type='nf4')
tok = AutoTokenizer.from_pretrained(MODELE)
model = AutoModelForCausalLM.from_pretrained(
    MODELE, quantization_config=quant, device_map='auto')
model.eval()
print('charge')

## 4. Les prompts

Les mêmes que le banc, en écriture arabe. Ils portent sur des sujets du
quotidien — c'est ce qui produit le registre voulu.

La consigne demande de la **fusha**, ce qui donne l'étiquette.

In [ ]:
PROMPTS = [
 {
  "id": "tn-001",
  "categorie": "quotidien",
  "texte": "احكيلي شنوة تعمل في نهار عادي متاعك، من الصباح للعشية"
 },
 {
  "id": "tn-002",
  "categorie": "quotidien",
  "texte": "شنوة اللي يعجبك في تونس و شنوة اللي ما يعجبكش؟"
 },
 {
  "id": "tn-003",
  "categorie": "quotidien",
  "texte": "احكيلي على آخر مرة ضحكت فيها برشا، شنوة صار وقتها؟"
 },
 {
  "id": "tn-004",
  "categorie": "quotidien",
  "texte": "كيفاش تقضي الويكاند متاعك في العادة؟ احكيلي بالتفصيل"
 },
 {
  "id": "tn-005",
  "categorie": "quotidien",
  "texte": "شنوة الماكلة اللي تحبها اكثر من غيرها و علاش؟"
 },
 {
  "id": "tn-006",
  "categorie": "explication",
  "texte": "علاش الناس في تونس يشربو القهوة برشا؟ فسرلي"
 },
 {
  "id": "tn-007",
  "categorie": "explication",
  "texte": "شنوة الفرق بين الصيف و الشتاء في تونس؟ احكيلي على الزوز"
 },
 {
  "id": "tn-008",
  "categorie": "explication",
  "texte": "فسرلي علاش ماء البحر مالح و ماء الواد لا"
 },
 {
  "id": "tn-009",
  "categorie": "explication",
  "texte": "علاش المواصلات في تونس فيها مشاكل في رايك؟"
 },
 {
  "id": "tn-010",
  "categorie": "explication",
  "texte": "شنوة معناها الاقتصاد الموازي؟ فسرهالي بكلام بسيط"
 },
 {
  "id": "tn-011",
  "categorie": "explication",
  "texte": "علاش الناس تنسى الاحلام متاعها كي تفيق من النوم؟"
 },
 {
  "id": "tn-012",
  "categorie": "explication",
  "texte": "فسرلي كيفاش تخدم الانترنت، بكلام يفهمو الجميع"
 },
 {
  "id": "tn-013",
  "categorie": "procedure",
  "texte": "كيفاش نطيب كسكسي تونسي؟ اعطيني الخطوات وحدة وحدة"
 },
 {
  "id": "tn-014",
  "categorie": "procedure",
  "texte": "كيفاش نبدل عجلة الكرهبة كي تتفشش؟ فسرلي بالتفصيل"
 },
 {
  "id": "tn-015",
  "categorie": "procedure",
  "texte": "عندي نبتة في الدار و ذبلت. شنوة نعمل باش نرجعها؟"
 },
 {
  "id": "tn-016",
  "categorie": "procedure",
  "texte": "كيفاش نحضر روحي مليح لمقابلة خدمة؟"
 },
 {
  "id": "tn-017",
  "categorie": "procedure",
  "texte": "اعطيني طريقة باش ننظم الوقت متاعي في النهار"
 },
 {
  "id": "tn-018",
  "categorie": "conseil",
  "texte": "اعطيني نصيحة باش نتعلم لغة جديدة و انا ما عنديش برشا وقت"
 },
 {
  "id": "tn-019",
  "categorie": "conseil",
  "texte": "خويا الصغير ما يحبش يقرا. شنوة نعمل معاه؟"
 },
 {
  "id": "tn-020",
  "categorie": "conseil",
  "texte": "نحب نبدا مشروع صغير و ما عنديش برشا فلوس. من وين نبدا؟"
 },
 {
  "id": "tn-021",
  "categorie": "conseil",
  "texte": "شنوة تنصحني نزور في تونس اذا عندي زوز ايام برك؟"
 },
 {
  "id": "tn-022",
  "categorie": "conseil",
  "texte": "صاحبي زعلان مني و ما نعرفش علاش. كيفاش نحكي معاه؟"
 },
 {
  "id": "tn-023",
  "categorie": "conseil",
  "texte": "نحس روحي تعبان برشا في الخدمة. شنوة تنصحني نعمل؟"
 },
 {
  "id": "tn-024",
  "categorie": "recit",
  "texte": "احكيلي حكاية قصيرة على ولد ضاع في المدينة القديمة"
 },
 {
  "id": "tn-025",
  "categorie": "recit",
  "texte": "تخيل روحك قاعد في مقهى في سيدي بوسعيد. وصفلي شنوة تشوف"
 },
 {
  "id": "tn-026",
  "categorie": "recit",
  "texte": "احكيلي حكاية على جدة تحكي لحفيدها على زمان بكري"
 },
 {
  "id": "tn-027",
  "categorie": "recit",
  "texte": "اكتبلي وصف لسوق نهار الاحد، بالحس و الروائح متاعو"
 },
 {
  "id": "tn-028",
  "categorie": "recit",
  "texte": "تخيل بلدة صغيرة ما فماش فيها ضو. كيفاش تعيش الناس فيها؟"
 },
 {
  "id": "tn-029",
  "categorie": "traduction",
  "texte": "ترجملي هالجملة للتونسي و فسرلي اختياراتك : « Je ne sais pas si je vais venir demain, ça dépend du travail »"
 },
 {
  "id": "tn-030",
  "categorie": "traduction",
  "texte": "كيفاش تقول بالتونسي « Il pleut des cordes » ؟ و علاش هكاكا؟"
 },
 {
  "id": "tn-031",
  "categorie": "traduction",
  "texte": "ترجم للتونسي و فسرلي : « Nous devons trouver une solution avant la fin du mois »"
 },
 {
  "id": "tn-032",
  "categorie": "traduction",
  "texte": "شنوة الفرق بين « برشا » و « شوية » ؟ اعطيني امثلة"
 },
 {
  "id": "tn-033",
  "categorie": "registre",
  "texte": "اكتبلي رسالة قصيرة لصاحبي باش نعتذرلو على تاخيرة"
 },
 {
  "id": "tn-034",
  "categorie": "registre",
  "texte": "فسرلي شنوة الفرق بين اللي يتكلم تونسي و اللي يتكلم فصحى"
 }
]

CONSIGNE = (
    'أجب دائماً باللغة العربية الفصحى الحديثة، بأسلوب واضح ومباشر. '
    'لا تستعمل أي لهجة عامية.'
)

print(len(PROMPTS), 'prompts')

## 5. Générer

`ECHANTILLONS` réponses par prompt. Avec 34 prompts, 12 échantillons
donnent ~400 réponses, soit largement les 1 800 blocs visés.

Le fichier est réécrit à chaque prompt : une déconnexion de Colab ne perd
que le prompt en cours. **Relancer la cellule reprend où elle s'est
arrêtée.**

In [ ]:
import json, os, time

ECHANTILLONS = 12
SORTIE = 'llm_fusha.jsonl'

deja = set()
if os.path.exists(SORTIE):
    for ligne in open(SORTIE, encoding='utf-8'):
        deja.add(json.loads(ligne)['cle'])
    print(len(deja), 'reponses deja produites — reprise')

debut = time.time()
with open(SORTIE, 'a', encoding='utf-8') as fh:
    for i, p in enumerate(PROMPTS, 1):
        manquants = [k for k in range(ECHANTILLONS) if f"{p['id']}-{k}" not in deja]
        if not manquants:
            continue
        msgs = [{'role': 'system', 'content': CONSIGNE},
                {'role': 'user', 'content': p['texte']}]

        # `return_dict=True` rend un dictionnaire : c'est la forme stable
        # entre versions de transformers. Sans lui, les versions recentes
        # rendent un BatchEncoding et `entree.shape` leve un AttributeError.
        enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                      return_tensors='pt', return_dict=True)
        enc = {k: v.to(model.device) for k, v in enc.items()}
        n_entree = enc['input_ids'].shape[-1]

        sorties = model.generate(**enc, max_new_tokens=400, do_sample=True,
                                 temperature=0.9, top_p=0.95,
                                 num_return_sequences=len(manquants),
                                 pad_token_id=tok.eos_token_id)

        for k, s in zip(manquants, sorties):
            texte = tok.decode(s[n_entree:], skip_special_tokens=True).strip()
            fh.write(json.dumps({'cle': f"{p['id']}-{k}", 'prompt_id': p['id'],
                                 'modele': MODELE, 'texte': texte},
                                ensure_ascii=False) + '\n')
        fh.flush()
        ecoule = time.time() - debut
        print(f"{i}/{len(PROMPTS)}  {p['id']}  +{len(manquants)} reponses  "
              f"({ecoule/60:.1f} min ecoulees)")
print('termine')

## 6. Vérifier le volume

L'objectif est **~1 800 blocs**, soit environ 108 000 mots. En dessous de
1 000, remontez `ECHANTILLONS` et relancez la cellule 5 — elle reprend.

In [ ]:
import json

mots = 0
n = 0
for ligne in open('llm_fusha.jsonl', encoding='utf-8'):
    mots += len(json.loads(ligne)['texte'].split())
    n += 1
print(f'{n} reponses, {mots} mots -> ~{mots//60} blocs de 60 mots')
print('objectif : 1800 blocs')

## 7. Récupérer le fichier

Le téléchargement démarre tout seul. Rangez le fichier dans le dépôt sous
`apps/darija-bench/results/`, puis dites-le — la suite se fait en local.

In [ ]:
from google.colab import files
files.download('llm_fusha.jsonl')